# QDSV/QIntent Batch Test for qLDPC-Style Post-Decoding

This notebook is the next step after the single-syndrome qLDPC-style demo.

It builds a public-safe sparse-check toy code, generates multiple observed syndromes, creates correction candidates with a simple maximum-likelihood / minimum-weight decoder baseline, and then uses QIntent structured semantic scoring to rank the candidate corrections.

The goal is not to claim a full qLDPC decoder benchmark. The goal is to test the post-decoding role of QDSV: given decoder-generated candidate hypotheses, can a structured semantic decision layer prefer lower-risk correction paths while preserving an auditable trace?

## 1. Install QIntent

This notebook installs QIntent from GitHub main so it can run before the next PyPI release.

In [ ]:
!pip install -q --upgrade "git+https://github.com/qdsvquantum-afk/qintent.git"

In [ ]:
import json
import math
from itertools import combinations
from pathlib import Path

import pandas as pd

from qintent import QIntentClient

API_URL = "https://qdsv-api-568788785178.us-central1.run.app/api"
client = QIntentClient(api_url=API_URL, timeout=60)

spec = client.spec()
spec["public_limits"]

## 2. Define a sparse qLDPC-style toy structure

`check_columns` represents a small sparse syndrome map. Each qubit column indicates which checks would fire if that qubit had an error.

`logical_sensitive_qubits` marks qubits that we treat as higher logical-risk areas in this public-safe toy experiment.

The selected single-qubit true errors are intentionally chosen so the minimum-weight decoder baseline has a high-confidence candidate, while the candidate set also contains alternative syndrome-compatible corrections with lower logical risk.

In [ ]:
check_columns = {
    0: (1, 0, 1, 0, 0),
    1: (1, 1, 0, 0, 0),
    2: (1, 0, 0, 1, 0),
    3: (0, 1, 0, 0, 1),
    4: (0, 0, 1, 1, 0),
    5: (0, 1, 0, 1, 1),
    6: (1, 0, 1, 1, 0),
    7: (0, 0, 1, 0, 1),
    8: (1, 1, 0, 1, 1),
    9: (0, 1, 1, 0, 0),
    10: (0, 0, 0, 1, 1),
    11: (1, 1, 0, 0, 1),
}

logical_sensitive_qubits = {1, 6, 8, 10, 11}
true_error_scenarios = [(1,), (6,), (8,), (10,), (11,)]
max_candidate_weight = 3
physical_error_rate = 0.08

pd.DataFrame([
    {"qubit": q, "check_column": "".join(str(bit) for bit in column), "logical_sensitive": q in logical_sensitive_qubits}
    for q, column in check_columns.items()
])

## 3. Candidate generator and decoder baseline

For each observed syndrome, the toy decoder enumerates low-weight correction hypotheses whose predicted syndrome matches the observation. The baseline selects the highest `decoder_confidence`, which is a normalized likelihood proxy that favors lower-weight candidates.

QDSV/QIntent then ranks the same candidate set using structured evidence and risk.

In [ ]:
def xor_syndrome(qubits):
    out = [0] * len(next(iter(check_columns.values())))
    for q in qubits:
        out = [a ^ b for a, b in zip(out, check_columns[q])]
    return tuple(out)


def generate_candidates_for_syndrome(scenario_id, observed_syndrome):
    raw = []
    for weight in range(1, max_candidate_weight + 1):
        for qubits in combinations(check_columns.keys(), weight):
            predicted = xor_syndrome(qubits)
            if predicted != observed_syndrome:
                continue

            logical_overlap = sum(q in logical_sensitive_qubits for q in qubits)
            log_likelihood = weight * math.log(physical_error_rate) + (len(check_columns) - weight) * math.log(1 - physical_error_rate)
            raw.append((qubits, weight, logical_overlap, log_likelihood, predicted))

    if not raw:
        return []

    likelihoods = [item[3] for item in raw]
    low, high = min(likelihoods), max(likelihoods)

    rows = []
    for candidate_index, (qubits, weight, logical_overlap, log_likelihood, predicted) in enumerate(raw):
        decoder_confidence = 1000 if high == low else int(round(600 + 400 * (log_likelihood - low) / (high - low)))

        rows.append({
            "scenario_id": scenario_id,
            "candidate_index": candidate_index,
            "correction_qubits": "".join(str(q) for q in qubits),
            "candidate_weight": weight,
            "observed_syndrome": "".join(str(bit) for bit in observed_syndrome),
            "predicted_syndrome": "".join(str(bit) for bit in predicted),
            "syndrome_support": max(0, 1000 - 20 * max(0, weight - 1)),
            "check_consistency": 1000,
            "logical_preservation": max(0, 960 - 260 * logical_overlap - 25 * max(0, weight - 1)),
            "distance_safety": max(0, 940 - 220 * logical_overlap - 20 * weight),
            "decoder_confidence": decoder_confidence,
            "propagation_safety": max(0, 940 - 60 * weight - 70 * logical_overlap),
            "syndrome_risk": min(1000, 35 + 12 * weight),
            "logical_risk": min(1000, 25 + 230 * logical_overlap + 20 * weight),
            "syndrome_entropy_adjustment": 20 if weight == 2 and logical_overlap == 0 else (-20 if logical_overlap else 0),
        })
    return rows


scenario_rows = []
scenario_metadata = []
for scenario_id, true_error in enumerate(true_error_scenarios):
    observed = xor_syndrome(true_error)
    rows = generate_candidates_for_syndrome(scenario_id, observed)
    scenario_rows.append(rows)
    scenario_metadata.append({
        "scenario_id": scenario_id,
        "true_error_qubits": "".join(str(q) for q in true_error),
        "observed_syndrome": "".join(str(bit) for bit in observed),
        "candidate_count": len(rows),
    })

pd.DataFrame(scenario_metadata)

## 4. QIntent structured semantic score declaration

In [ ]:
source = """
find_rows("candidate_index")
  .using_structured_semantic_score([
      block("syndrome", [
          signal("syndrome_support", influence=30, priority=2),
          signal("check_consistency", influence=20, priority=1),
      ], influence=30, priority=2, risk_adjustment="syndrome_risk", adjustments=[
          adjustment("syndrome_entropy_adjustment", influence=5),
      ]),
      block("logical_safety", [
          signal("logical_preservation", influence=40, priority=3),
          signal("distance_safety", influence=20, priority=2),
      ], influence=40, priority=3, risk_adjustment="logical_risk"),
      block("decoder", [
          signal("decoder_confidence", influence=25, priority=1),
          signal("propagation_safety", influence=15, priority=2),
      ], influence=30, priority=1),
  ], global_risk="logical_risk", profile="qldpc_post_decoding_batch")
  .accept_if(threshold=600)
  .rank()
  .top_k(1)
"""

print(source)

## 5. Explain one scenario before batch execution

The public evidence should confirm that QDSV exposes the operation structure and trace, not the private formula.

In [ ]:
passport = client.explain(source, rows=scenario_rows[0])
predicate_summary = passport["semantic_execution_passport"]["predicate"]

assert predicate_summary["kind"] == "structured_semantic_score"
assert predicate_summary["blocks_count"] == 3
assert predicate_summary["signals_count"] == 6
assert predicate_summary["internal_formula_exposed"] is False

predicate_summary

## 6. Run the batch comparison

This cell performs one QIntent `run()` call per scenario. Keep the scenario count small for the public-preview API quota.

In [ ]:
summary_rows = []
selected_by_scenario = []

for meta, rows in zip(scenario_metadata, scenario_rows):
    candidates = pd.DataFrame(rows)
    baseline = candidates.sort_values("decoder_confidence", ascending=False).iloc[0]

    result = client.run(source, rows=rows)
    selected = result["result"]["selected_rows"][0]
    selected_by_scenario.append(selected)

    summary_rows.append({
        "scenario_id": meta["scenario_id"],
        "true_error_qubits": meta["true_error_qubits"],
        "observed_syndrome": meta["observed_syndrome"],
        "candidate_count": meta["candidate_count"],
        "baseline_candidate": int(baseline["candidate_index"]),
        "baseline_qubits": baseline["correction_qubits"],
        "baseline_confidence": int(baseline["decoder_confidence"]),
        "baseline_logical_risk": int(baseline["logical_risk"]),
        "qdsv_candidate": int(selected["candidate_index"]),
        "qdsv_qubits": selected["correction_qubits"],
        "qdsv_confidence": int(selected["decoder_confidence"]),
        "qdsv_logical_risk": int(selected["logical_risk"]),
        "risk_reduction": int(baseline["logical_risk"]) - int(selected["logical_risk"]),
    })

summary = pd.DataFrame(summary_rows)
summary

## 7. Aggregate metrics

In [ ]:
metrics = {
    "scenarios": int(len(summary)),
    "baseline_avg_logical_risk": float(summary["baseline_logical_risk"].mean()),
    "qdsv_avg_logical_risk": float(summary["qdsv_logical_risk"].mean()),
    "avg_risk_reduction": float(summary["risk_reduction"].mean()),
    "risk_reduction_positive_count": int((summary["risk_reduction"] > 0).sum()),
}

metrics

## 8. Inspect one scenario in detail

In [ ]:
scenario_id = 0
pd.DataFrame(scenario_rows[scenario_id]).sort_values("decoder_confidence", ascending=False)

In [ ]:
print(json.dumps(selected_by_scenario[scenario_id]["_qdsv_decision_model"], indent=2))

## 9. Save evidence artifacts

In [ ]:
evidence = {
    "api_url": API_URL,
    "profile": "qldpc_post_decoding_batch",
    "toy_sparse_check_structure": {
        "check_columns": {str(k): list(v) for k, v in check_columns.items()},
        "logical_sensitive_qubits": sorted(logical_sensitive_qubits),
        "true_error_scenarios": [list(item) for item in true_error_scenarios],
        "max_candidate_weight": max_candidate_weight,
        "physical_error_rate": physical_error_rate,
        "candidate_generation": "enumerate syndrome-compatible low-weight correction hypotheses",
    },
    "qintent_source": source,
    "predicate_summary": predicate_summary,
    "scenario_metadata": scenario_metadata,
    "summary": summary.to_dict(orient="records"),
    "metrics": metrics,
    "selected_by_scenario": selected_by_scenario,
    "internal_formula_exposed": False,
}

Path("qdsv_qldpc_batch_decoder_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
summary.to_csv("qdsv_qldpc_batch_decoder_summary.csv", index=False)

print("Saved:")
print("- qdsv_qldpc_batch_decoder_evidence.json")
print("- qdsv_qldpc_batch_decoder_summary.csv")

## Interpretation

This batch test evaluates QDSV/QIntent as a post-decoding decision layer over decoder-generated candidates.

The baseline decoder confidence favors lower-weight, higher-likelihood candidates. QDSV can select a different syndrome-compatible correction when the structured semantic evidence indicates lower logical risk and stronger logical safety.

This is still not a formal qLDPC benchmark. The next step is to replace the toy decoder with a real qLDPC decoder or predecoder output and compare against standard decoding metrics.